# ManimFlow

Autonomous animation generator powered by [SWE-agent](https://github.com/SWE-agent/SWE-agent) — **no API keys required**.

## Usage
1. Runtime → Run All
2. Describe your animation in the text box that appears in the last cell
3. Click **Generate**

In [ ]:
import importlib.util, pathlib, sys, subprocess, shutil, platform

if platform.system() == 'Linux':
    if subprocess.run(['dpkg', '-s', 'libpango1.0-dev'], capture_output=True).returncode != 0:
        subprocess.run(['apt-get', 'install', '-y', '-q', 'libcairo2-dev', 'libpango1.0-dev', 'ffmpeg'],
                       capture_output=True)

if not importlib.util.find_spec('manim'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'manim', '-q'], check=True)

if not pathlib.Path('SWE-agent/pyproject.toml').exists():
    subprocess.run(['rm', '-rf', 'SWE-agent'])
    subprocess.run(['git', 'clone', 'https://github.com/SWE-agent/SWE-agent.git',
                    '--depth=1', '--branch', 'v1.1.0'], capture_output=True)

if not importlib.util.find_spec('sweagent'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'SWE-agent', '-q'], check=True)

swa_path = str(pathlib.Path('SWE-agent').resolve())
if swa_path not in sys.path:
    sys.path.insert(0, swa_path)

import sweagent
print(f'sweagent ready: {sweagent.__file__}')


In [ ]:
import pathlib, subprocess, re, json, threading, urllib.request
from http.server import HTTPServer, BaseHTTPRequestHandler
from socketserver import ThreadingMixIn

BASE = pathlib.Path.cwd()
MANIM_OUTPUT = BASE / 'manim_output'
MANIM_DOCS = BASE / 'manim_docs'
MANIM_OUTPUT.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'bash', '-c',
    f'cd {MANIM_OUTPUT} && '
    'git init -q && '
    'git config user.email "agent@colab" && '
    'git config user.name "agent" && '
    'git commit --allow-empty -q -m init 2>/dev/null || true'
])

if not MANIM_DOCS.exists():
    subprocess.run(
        f'git clone --depth=1 --filter=blob:none --sparse '
        f'https://github.com/ManimCommunity/manim.git {MANIM_DOCS} 2>&1 | tail -2 && '
        f'cd {MANIM_DOCS} && git sparse-checkout set docs 2>&1 | tail -1',
        shell=True
    )
    print('Manim docs cloned')
else:
    print('Manim docs already present')

EXA_URL = "https://demos.exa.ai/chatbot-demo/api/chat/stream"
AGENT_SYSTEM_PROMPT = """\
You are a precise assistant. Follow the user's instructions exactly.
- Never add language tags to code fences — use plain triple backticks only: ``` not ```python or ```bash
- Never use heredocs (cat << EOF). Write files with python3 -c instead.
- Write your entire shell command on a single line — no literal newlines inside the code fence.
- No follow-up suggestions, no lists of questions. Be concise and direct."""

def _fix_encoding(text):
    return re.sub(r'Â([\x80-\xBF])', lambda m: m.group(1), text)

def _strip_followups(text):
    idx = text.find("\n\n```followups")
    if idx >= 0:
        text = text[:idx]
    text = re.sub(r'\n\[(["\']).*?\1(?:,\s*(["\']).*?\2)*\s*\]\s*$', '', text, flags=re.DOTALL)
    return text.rstrip()

def _strip_fence_tags(text):
    text = re.sub(r'^`{3,}[a-zA-Z_][a-zA-Z0-9_]*$', '```', text, flags=re.MULTILINE)
    text = re.sub(r'^`{4,}$', '```', text, flags=re.MULTILINE)
    return text

def _clean_code_fences(text):
    last = text.rfind("```\n")
    if last == -1:
        return text
    before = text[:last]
    after = text[last + 4:]
    close = after.rfind("\n```")
    if close == -1:
        return text
    body = after[:close]
    rest = after[close + 4:]
    body = re.sub(r'^`+\n?', '', body)
    if re.search(r'python3\s+-c', body):
        body = body.replace('\n', '\\n')
    else:
        body = body.split('\n')[0]
    return before + "```\n" + body + "\n```" + rest

def _rewrite_heredocs(text):
    def replace(m):
        filepath = m.group(2)
        content = m.group(3)
        escaped = content.replace('\\', '\\\\').replace("'", "\\'").replace('\n', '\\n')
        return (f"python3 -c \"import pathlib; "
                f"pathlib.Path('{filepath}').parent.mkdir(parents=True, exist_ok=True); "
                f"pathlib.Path('{filepath}').write_text('{escaped}')\"")
    return re.sub(r"cat\s*<<\s*['\"]?(\w+)['\"]?\s*>\s*(\S+)\n([\s\S]*?)\n\1", replace, text)

def _post_process(text):
    text = _fix_encoding(text)
    text = _strip_followups(text)
    text = _strip_fence_tags(text)
    text = _rewrite_heredocs(text)
    text = _clean_code_fences(text)
    return text

def _collect_exa_stream(messages):
    system_msg = next((m for m in messages if m['role'] == 'system'), None)
    non_system = [m for m in messages if m['role'] != 'system']
    last = non_system[-1]
    history = [{'role': m['role'], 'content': m['content']} for m in non_system[:-1]]
    system_content = system_msg['content'] if system_msg else AGENT_SYSTEM_PROMPT
    payload = json.dumps({
        'message': f"{system_content}\n\n{last['content']}",
        'history': history,
        'exaEnabled': False,
        'model': 'google/gemini-2.5-flash',
        'searchType': 'instant',
    }).encode()
    req = urllib.request.Request(EXA_URL, data=payload,
        headers={'Content-Type': 'application/json', 'Accept': 'text/event-stream'})
    full = ''
    current_event = None
    with urllib.request.urlopen(req, timeout=120) as resp:
        buf = b''
        for raw in resp:
            buf += raw
            lines = buf.split(b'\n')
            buf = lines.pop()
            for line in lines:
                t = line.decode('utf-8', errors='replace').strip()
                if not t:
                    continue
                if t.startswith('event:'):
                    current_event = t[6:].strip()
                elif t.startswith('data:') and current_event == 'content':
                    try:
                        full += json.loads(t[5:].strip()).get('content', '')
                    except Exception:
                        pass
    return _post_process(full)

LOCAL_PORT = 18642

class _ThreadingServer(ThreadingMixIn, HTTPServer):
    daemon_threads = True

class _Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_GET(self):
        if self.path == '/v1/models':
            body = json.dumps({'object': 'list', 'data': [
                {'id': 'manimator', 'object': 'model', 'created': 0, 'owned_by': 'exa'}
            ]}).encode()
            self.send_response(200)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path != '/v1/chat/completions':
            self.send_response(404)
            self.end_headers()
            return
        length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(length))
        messages = body.get('messages', [])
        try:
            content = _collect_exa_stream(messages)
        except Exception as e:
            self.send_response(500)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(json.dumps({'error': str(e)}).encode())
            return
        cid = 'chatcmpl-local'
        resp_body = json.dumps({
            'id': cid,
            'object': 'chat.completion',
            'model': 'manimator',
            'choices': [{
                'index': 0,
                'message': {'role': 'assistant', 'content': content},
                'finish_reason': 'stop',
            }],
            'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0},
        }).encode()
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(resp_body)))
        self.end_headers()
        self.wfile.write(resp_body)

def _start_server():
    server = _ThreadingServer(('127.0.0.1', LOCAL_PORT), _Handler)
    t = threading.Thread(target=server.serve_forever, daemon=True)
    t.start()
    return server

_start_server()
print(f'Local LLM server running on port {LOCAL_PORT}')

MANIMATOR_CONFIG = f"""\
agent:
  model:
    name: "openai/manimator"
    api_base: "http://127.0.0.1:{LOCAL_PORT}/v1"
    api_key: "dummy"
    max_input_tokens: 128000
    max_output_tokens: 4096
    per_instance_cost_limit: 0.0
    total_cost_limit: 0.0
  history_processors:
    - type: last_n_observations
      n: 3
  templates:
    system_template: |-
      You are a Manim animation coding agent. You receive a scene plan and must implement it as a working Manim animation.

      DOCS: Manim reference docs are at {MANIM_DOCS}/docs/source/
      - Search: grep -r "ClassName" {MANIM_DOCS}/docs/source/ --include="*.rst" -l
      - Read:   cat {MANIM_DOCS}/docs/source/reference/<file>.rst

      RULES:
      - Use `python3 -m manim` — never `manim`
      - Script goes in {MANIM_OUTPUT}/scene.py
      - Write files with: python3 -c "import pathlib; pathlib.Path('f.py').write_text(chr(39)*3 + code + chr(39)*3)"
      - Render with: python3 -m manim -pql {MANIM_OUTPUT}/scene.py <ClassName> 2>&1
      - The scene class must subclass Scene
      - If render fails, read the error and fix — keep trying
      - Stop as soon as you see "File ready at" — output exactly: DONE
      - Do NOT render multiple scenes. ONE script, ONE class, ONE render.
      - Do NOT ask for input. Do NOT explain. Just code and run.

      RESPONSE FORMAT — always exactly this shape, nothing else:
      DISCUSSION
      one line of reasoning
      ```
      one_shell_command
      ```
    instance_template: |-
      Implement this animation plan as a Manim scene named AnimScene in {MANIM_OUTPUT}, then render it.

      PLAN:
      {{{{problem_statement}}}}

      Work in: {MANIM_OUTPUT}
    next_step_template: |-
      OBSERVATION:
      {{{{observation}}}}
    next_step_no_output_template: |-
      Command ran with no output.
  tools:
    parse_function:
      type: "thought_action"
    env_variables:
      PAGER: cat
      MANPAGER: cat
      GIT_PAGER: cat
    reset_commands:
      - "set +H"
"""

pathlib.Path('SWE-agent/config/manimator.yaml').write_text(MANIMATOR_CONFIG)
print('Setup complete')


In [ ]:
import subprocess, pathlib, shutil, requests, threading, time, os
import IPython.display

def call_llm(messages):
    r = requests.post(
        f"http://127.0.0.1:{LOCAL_PORT}/v1/chat/completions",
        json={"model": "manimator", "messages": messages, "max_tokens": 1024},
        headers={"Authorization": "Bearer dummy", "Content-Type": "application/json"},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

def plan_animation(user_prompt):
    return call_llm([
        {
            "role": "system",
            "content": (
                "You are a Manim animation planner. Given a topic, write a concise scene plan "
                "for a short animation — 3 to 5 scenes max. For each scene describe: what shapes/colors appear, "
                "how they move, what text labels show. Be brief and specific. No prose, no markdown headers, "
                "just a numbered list. Total response under 200 words."
            ),
        },
        {"role": "user", "content": user_prompt},
    ])

def run_manimator(user_prompt):
    print(f"Planning: {user_prompt!r}")
    plan = plan_animation(user_prompt)
    print("\n=== Scene Plan ===")
    print(plan)
    print("=" * 40)

    for f in MANIM_OUTPUT.glob('*.py'):
        f.unlink()

    task_file = BASE / 'task.md'
    task_file.write_text(plan)

    os.environ['LITELLM_REQUEST_TIMEOUT'] = '120'
    proc = subprocess.Popen(
        [
            "python3", "-m", "sweagent", "run",
            "--config", "SWE-agent/config/manimator.yaml",
            "--env.deployment.type=local",
            "--env.repo.type=preexisting",
            "--env.repo.repo_name=manim_output",
            "--agent.max_requeries=1",
            f"--problem_statement.path={task_file}",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, 'PYTHONPATH': str(pathlib.Path('SWE-agent').resolve())},
        text=True,
        bufsize=1,
    )

    print("\n=== Agent Log ===")
    last_line_time = [time.time()]
    done = [False]

    def heartbeat():
        while not done[0]:
            time.sleep(1)
            if time.time() - last_line_time[0] > 10:
                print("  ⏳ waiting for LLM response...", flush=True)
                last_line_time[0] = time.time()

    t = threading.Thread(target=heartbeat, daemon=True)
    t.start()

    for line in proc.stdout:
        line = line.rstrip()
        last_line_time[0] = time.time()
        print(line)
        if "File ready at '" in line or "Exit status" in line:
            proc.terminate()
            break
    done[0] = True
    proc.wait()

    videos = sorted(BASE.rglob('*.mp4'), key=lambda p: p.stat().st_mtime)
    if not videos:
        print("\nNo video rendered.")
        return

    final = videos[-1]
    dest = BASE / 'animation.mp4'
    shutil.copy(final, dest)

    for mp4 in videos:
        if mp4 != dest:
            try: mp4.unlink()
            except: pass
    for d in BASE.rglob('media'):
        if d.is_dir():
            shutil.rmtree(d, ignore_errors=True)

    print(f"\nFinal MP4: {dest}")
    IPython.display.display(
        IPython.display.Video(str(dest), embed=True, html_attributes='controls width="720"')
    )

print("run_manimator() ready")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

prompt_box = widgets.Textarea(
    placeholder="e.g. Explain how a Fourier series builds up a square wave",
    layout=widgets.Layout(width="700px", height="100px"),
)
btn = widgets.Button(description="Generate", button_style="primary")
status = widgets.Output()

def on_generate(b):
    with status:
        status.clear_output()
        if not prompt_box.value.strip():
            print("Please enter a prompt.")
            return
        run_manimator(prompt_box.value.strip())

btn.on_click(on_generate)
display(prompt_box, btn, status)
